> Graphs built out of signals and/or windows thereof to try and identify events of interest

In [ ]:
import pathlib

import pandas as pd

import plotly
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.plotting import show

# from bokeh.io import export_png
# from IPython.display import Image

import trapallada.plot
import trapallada.analysis

import ipywidgets as widgets
from IPython.display import clear_output

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
output_notebook(resources=INLINE)

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

# Input data

Particular *sub*directories with
- the trained model, and
- the corresponding graph. 
They should be a `str` so that parameters can be injected with papermill.

In [ ]:
nodes_file = '../results/jesus/starwVanillaZ05_00/graphs/sigsum/sigsum_nodes.csv'
edges_file = '../results/jesus/starwVanillaZ05_00/graphs/sigsum/sigsum_edges.csv'
analysis_file = '../results/cvae/min_count_rate_gti_with_background/fft_vanilla_z_5/analysis.parquet'
# tar_file = '/home/manu/Sync/git/trapallada/data/min_count_rate_gti_with_background/full_signals/0700381101.parquet'
# fit_file = 'P0700381101PNS001SRCTSR8001.FTZ'
# full_signals_dir = '/home/manu/Sync/git/trapallada/data/min_count_rate_gti_with_background/full_signals'
data_dir = '/home/manu/Sync/git/trapallada/data/min_count_rate_gti_with_background'
signal = '0700381101/P0700381101PNS001SRCTSR8001'
# main_type = 'X'
# x = 1.2
# y = 3.4

In [ ]:
nodes_df = pd.read_csv(nodes_file)
edges_df = pd.read_csv(edges_file)

In [ ]:
import papermill as pm

analysis_cols = ['signal', 'main_type', 'main_id', 'RA', 'DEC', 'CRA', 'CDEC']
nodes_cols = ['Id', 'leiden', 'x', 'y']
analysis_df = pd.read_parquet(analysis_file)
analyzed_signals_df =  analysis_df[analysis_cols].drop_duplicates().merge(
    nodes_df[nodes_cols], left_on='signal', right_on='Id', how='left'
)
analyzed_signals_df = analyzed_signals_df[analysis_cols + nodes_cols[1:]]
analyzed_signals_df.head()

In [ ]:
for i, row in analyzed_signals_df.iterrows():

    # trapallada.plot.full_signal(row['signal'], pathlib.Path(data_dir) / 'full_signals').show()
    # pm.execute_notebook(
    #     'path/to/input.ipynb',
    #     'path/to/output.ipynb',
    #     # parameters=dict(nodes_file=0.6, edges_file=0.1, signal='sdfa', data_dir='foo')
    #     parameters=dict(signal='sdfa', data_dir='foo')
    # )

    row.to_dict()
    break

In [ ]:
row.to_dict()

In [ ]:
graph_plot = trapallada.analysis.SignalsSuperGraph(nodes_df, edges_df, data_dir)
fig = graph_plot.make_fig()
show(fig)
# static_graph_file = 'graph.png'
# export_png(fig, filename=static_graph_file)
# Image(filename=static_graph_file)

In [ ]:
sorted_signals = analyzed_signals_df['signal'].sort_values()

signal_dropdown = widgets.Dropdown(
    options=sorted_signals,
    value=sorted_signals.iloc[0],
    description='Signal:',
    disabled=False
)

def plot_signal(change):
    
    clear_output(wait=True)
    selected_signal = change['new']
    fig = trapallada.plot.full_signal(selected_signal, pathlib.Path(data_dir) / 'full_signals')
    display(fig)
    display(signal_dropdown)
    display(analyzed_signals_df[analyzed_signals_df['signal'] == selected_signal])

signal_dropdown.observe(plot_signal, names='value')
display(signal_dropdown)

In [ ]:
# fig_signal = trapallada.plot.full_signal(signal, pathlib.Path(data_dir) / 'full_signals')
# # static_signal_file = 'signal.png'
# # fig_signal.write_image(static_signal_file, width=1_200, scale=2)
# # Image(filename=static_signal_file)
# fig_signal

In [ ]:
# nodes_df.loc[signal]